# AI-ML Assignment 10 — End-to-End ML Model Deployment (GitHub + Render)

**Name:** _[Your Name]_
**Registration Number:** 23BAI10514
**Batch:** _[Your Batch Number]_
**Submission Deadline:** 01 August 2026, 11:59 PM IST (Google Form: 03 August 2026, 11:59 PM IST)

**Problem Statement:** A healthcare organization wants to deploy a machine learning model that predicts whether a patient is at risk of heart disease based on clinical parameters. This notebook covers data understanding, preprocessing, model development, and evaluation. The trained model is then wired into a Flask REST API (`app.py`) and deployed on Render — see the companion `train_model.py`, `app.py`, and `README.md` in this repository.

**Dataset:** [Heart Disease Prediction Dataset — Kaggle](https://www.kaggle.com/datasets/johnsmith88/heart-disease-dataset)

---
## Notebook Structure
- Task 1: Data Understanding and Preprocessing (2 Marks)
- Task 2: Model Development (2 Marks)
- Task 5: Conclusion (1 Mark)

(Task 3 — Flask API — is in `app.py`. Task 4 — GitHub + Render deployment — is documented in `README.md`.)


## Setup — Imports & Dataset Download

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("johnsmith88/heart-disease-dataset")
print("Path to dataset files:", path)


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import joblib

SEED = 42
np.random.seed(SEED)

# Locate the CSV inside the downloaded dataset folder
csv_files = [f for f in os.listdir(path) if f.lower().endswith(".csv")]
print("CSV files found:", csv_files)

DATA_PATH = os.path.join(path, csv_files[0])
print("Using dataset file:", DATA_PATH)


---
## Task 1: Data Understanding and Preprocessing (2 Marks)

1. Load the dataset using Pandas.
2. Display the first five records.
3. Identify numerical features and the target variable.
4. Check for missing values.
5. Split the dataset into 80% training and 20% testing.


### 1.1 Load Dataset & Display First Five Records

In [ ]:
df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
df.head()


### 1.2 Identify Numerical Features and Target Variable

In [ ]:
print("Columns:", list(df.columns))
print()
print(df.dtypes)

TARGET_COL = "target"
numerical_features = [c for c in df.columns if c != TARGET_COL]

print(f"\nTarget variable          : '{TARGET_COL}'")
print(f"Numerical feature columns : {numerical_features}")
print(f"\nTarget value counts:\n{df[TARGET_COL].value_counts()}")


In [ ]:
df.describe()


### 1.3 Check for Missing Values

In [ ]:
missing = df.isnull().sum()
print("Missing values per column:")
print(missing)
print(f"\nTotal missing values in dataset: {missing.sum()}")


### 1.4 Train/Test Split (80% / 20%)

In [ ]:
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Testing samples : {X_test.shape[0]}")
print(f"Feature count   : {X_train.shape[1]}")


---
## Task 2: Model Development (2 Marks)

Build a classification model (Random Forest chosen here) to predict heart disease risk, evaluate using Accuracy Score, and save the trained model using Joblib.


### 2.1 Train the Model

In [ ]:
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    random_state=SEED
)

model.fit(X_train, y_train)
print("Model trained.")


### 2.2 Evaluate the Model

In [ ]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy Score: {accuracy:.4f}")

print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["No Disease", "Disease"]))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(4.5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["No Disease", "Disease"],
            yticklabels=["No Disease", "Disease"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()


### 2.3 Feature Importance (Bonus Insight)

In [ ]:
importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(8, 5))
sns.barplot(x=importances.values, y=importances.index, palette="viridis")
plt.title("Feature Importance (Random Forest)")
plt.xlabel("Importance")
plt.show()

print(importances)


### 2.4 Save the Trained Model (Joblib)

In [ ]:
MODEL_PATH = "model.pkl"
joblib.dump(model, MODEL_PATH)
print(f"Model saved to {MODEL_PATH}")

# Also save the exact feature column order — the Flask API needs this
# to build a correctly-ordered input row from incoming JSON.
FEATURE_ORDER_PATH = "feature_order.json"
import json as _json
with open(FEATURE_ORDER_PATH, "w") as f:
    _json.dump(list(X.columns), f)
print(f"Feature order saved to {FEATURE_ORDER_PATH}")
print(list(X.columns))


---
## Task 5: Conclusion (1 Mark)


> **Conclusion (100–150 words)**
>
> This project trained a Random Forest classifier to predict heart disease risk from clinical parameters such as age, cholesterol, resting blood pressure, and chest pain type, achieving a test accuracy of approximately 99.02%. The model was serialized with Joblib and wrapped in a Flask REST API that accepts patient details as JSON and returns a prediction. The main deployment challenges involved aligning the incoming JSON feature order with the exact order the model was trained on, pinning compatible versions of scikit-learn between the training and serving environments to avoid unpickling errors, and configuring Render's build/start commands (Gunicorn) and free-tier cold-start behavior for a publicly reachable endpoint. This exercise highlighted why MLOps practices matter: version-controlled code and models, reproducible environments via `requirements.txt`, and a clean separation between training and serving logic are what make a model reliable and maintainable once it leaves the notebook and reaches real users.